# TorchSpace tutorial — spatial debugging for PyTorch

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/torchspace/blob/main/TorchSpace_Tutorial.ipynb)

**TorchSpace** renders a PyTorch model as one interactive 3D scene:

| axis | meaning |
|---|---|
| **X · Y** | architecture (layers, blocks, skip connections) |
| **+Z** | forward activation magnitude (log scale) |
| **−Z** | backward gradient magnitude (log scale) |

Vanishing gradients, activation explosions, dead regions and forward/backward
asymmetry become *visible shapes* instead of numbers you have to hunt for.
TorchSpace is a non-invasive extension of
[torchview](https://github.com/mert-kurttutan/torchview): it copies none of its
code and keeps its full feature set available (see §5).

Everything below runs offline — the scene is self-contained HTML (three.js
inlined, statistics embedded), so it works in Colab, Jupyter, VS Code and as a
plain file. **No raw tensors ever leave your machine — only summary statistics.**

In [ ]:
try:
    import torchspace
except ImportError:
    %pip -q install --pre torchspace   # pre-release: --pre is required
    import torchspace
import torchview
print("torchspace ready — torchview", torchview.__version__)

## 1. Quick start: trace → run → look

Three lines of instrumentation around your normal forward/backward:

1. `trace(model, input_data=x)` records the architecture and installs
   statistics hooks (the model is *instrumented*, not modified).
2. Your next `model(x)` is captured as forward activation frames.
3. `run.capture_backward(loss)` runs `loss.backward()` and captures gradients.

`run.show()` renders the interactive scene inline. `run.detach()` removes all
hooks and returns the model to its untouched state.

In [ ]:
import torch, torch.nn as nn
import torchspace
from torchspace.demos import TinyResNet

torch.manual_seed(0)
model = TinyResNet().eval()
x = torch.randn(1, 3, 32, 32)

run = torchspace.trace(model, input_data=x)   # structure + instrumentation
out = model(x)                                # captured forward
run.capture_backward(out.sum())               # captured backward
run.detach()                                  # de-instrument

run.show(height=620)

### How to read the scene

* **drag** rotate · **wheel** zoom · **right-drag** pan
* **hover** a node → exact statistics tooltip
* **click** a node → inspector (right panel) *and the diagnostics panel
  (bottom-left) filters to that layer's findings* — "show all" clears it
* **double-click** a block → expand / collapse its children
* **2D Ortho** → torchview-style top-down architecture view
* **▶ Replay** → animate forward (green) / backward (purple) execution order
* the **ruler** beside the model gives the quantitative `1e^x` scale for both
  overlays; the **±Z scale** slider exaggerates bar heights

## 2. Diagnosing a pathological network

A 24-layer sigmoid MLP with bad initialization. Watch the −Z gradient staircase
shrink by ~9 orders of magnitude toward the input, one exploding layer spike in
+Z (red), and a mostly-dead ReLU stage. Every finding in the diagnostics panel
is rule-based and carries the evidence used — click one to jump to the layer.

In [ ]:
from torchspace.demos import PathologicalMLP

torch.manual_seed(0)
mlp = PathologicalMLP()
xb, yb = torch.randn(32, 64), torch.randint(0, 10, (32,))

run2 = torchspace.trace(mlp, input_data=xb)
loss = nn.CrossEntropyLoss()(mlp(xb), yb)
run2.capture_backward(loss)
run2.detach()

for w in run2.ir["warnings"][:8]:
    print(f"[{w['severity']:7s}] {w['rule']:26s} {w['node']}: {w['message']}")
print(f"... {len(run2.ir['warnings'])} findings total")

run2.show(height=620)

## 3. The numbers behind the scene: the IR

Everything the viewer shows lives in `run.ir` — a plain JSON-serializable dict
(nodes, edges, tensors, per-step statistic frames, diagnostics). It is the
stable contract between Python and the renderer, so you can post-process it,
diff it between runs, or store it next to your experiments.

In [ ]:
ir = run2.ir
act  = {f["node"]: f["stats"]["rms"] for f in ir["frames"]
        if f["kind"] == "activation"}
grad = {f["node"]: f["stats"]["rms"] for f in ir["frames"]
        if f["kind"] == "gradient"}

print(f"{'layer':<14} {'act rms':>12} {'grad rms':>12}")
for nid in sorted(act, key=lambda n: (len(n), n)):
    if nid.startswith("backbone.") and nid in grad:
        print(f"{nid:<14} {act[nid]:>12.3e} {grad[nid]:>12.3e}")

## 4. Architecture-only mode (zero memory, `meta` device)

`view()` gives torchview-class structural inspection without any runtime
capture — it even works on `device='meta'`, so you can inspect models too large
to materialize.

In [ ]:
structure = torchspace.view(TinyResNet(), input_size=(1, 3, 32, 32), device="meta")
structure.show(height=520)

## 5. torchview parity: classic graphviz output

TorchSpace traces at full detail, but torchview's own renderer stays available
at any granularity. `run.draw_graph(...)` / `run.export_dot(...)` accept every
`torchview.draw_graph` option (`depth`, `roll`, `graph_dir`,
`hide_inner_tensors`, `save_graph`, ...) and reuse the exact inputs of your
trace — capture hooks are suspended for this auxiliary pass.

In [ ]:
g = run.draw_graph(depth=2, roll=True, graph_dir="TB")  # run = TinyResNet from §1
g.visual_graph   # renders inline (Colab has the graphviz binary preinstalled)

## 6. Module attributes in the inspector

`trace()`/`view()` collect a `repr`-style summary of each module's public
attributes by default (`collect_attributes=False` to disable — see the privacy
note at the end). They appear in the viewer's inspector and in the IR:

In [ ]:
for n in run.ir["nodes"][:6]:
    if n.get("attrs"):
        print(f"{n['id']:<12} {n['attrs'][:90]}")

## 7. Instrumenting a real training loop

Hooks capture statistics on-device from your *actual* training steps — no
separate profiling pass. The viewer shows the most recent step; earlier steps
stay in `run.ir['frames']` (`step` field) for your own analysis.

In [ ]:
torch.manual_seed(0)
net = PathologicalMLP()
opt = torch.optim.SGD(net.parameters(), lr=0.05)
crit = nn.CrossEntropyLoss()

run3 = torchspace.trace(net, input_data=xb, mode="train")
for step in range(3):
    opt.zero_grad()
    loss = crit(net(xb), yb)          # captured forward
    run3.capture_backward(loss)       # backward + gradient capture
    opt.step()
    print(f"step {step}: loss {loss.item():.4f}")
run3.detach()                          # always de-instrument when done

run3.show(height=560)

## 8. Export & share

One self-contained HTML file — attach it to an issue, a PR review, or a report.
The IR JSON can be saved alongside for programmatic use.

In [ ]:
run2.export_html("pathological_mlp.torchspace.html")
run2.save_ir("pathological_mlp.torchspace.json")
print("wrote pathological_mlp.torchspace.{html,json}")

# In Colab, download them with:
# from google.colab import files
# files.download("pathological_mlp.torchspace.html")

---

### Privacy note

Exports contain **statistics only**, never raw tensors. Two edges to know
before sharing: attribute collection (§6) embeds `repr`-style summaries that
for custom modules can include paths or config values — use
`collect_attributes=False` for sensitive models; and a 1-element tensor's stats
(min = max = mean) necessarily reveal its exact value.

### Links

* Source & issues: `https://github.com/YOUR-GITHUB-USERNAME/torchspace`
* Built on [torchview](https://github.com/mert-kurttutan/torchview) (MIT) —
  no code copied; three.js r128 (MIT) is bundled in the viewer.
* License: MIT